# MolFrame + MolGFX: an interactive molecular workbench

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/miguelcsx/molgfx/blob/main/examples/workbench.ipynb)

This notebook installs the released `molframe` and `molgfx` packages and opens a live,
GPU-rendered view of haemoglobin (PDB 4HHB). The picture is drawn **in your browser** with
WebGPU; the kernel only sends the structure once and afterwards small semantic edits
(`ScenePatch`es). You can rotate (drag), zoom (wheel) and pick atoms (click), and type
commands in the console under the canvas.

Requirements: a browser with WebGPU (current Chrome or Edge; Safari and Firefox with WebGPU
enabled) and Python 3.12 or newer.

In [ ]:
%pip install -q --upgrade "molgfx[jupyter]"

In [ ]:
# Colab renders third-party widgets only after this call; elsewhere it does nothing.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

## Load a structure

`molframe` parses the file; `molgfx` borrows its coordinates without copying them.

In [ ]:
import importlib.metadata
import urllib.request

import molframe
import molgfx
from molgfx.viewer import Workbench

print({package: importlib.metadata.version(package) for package in ("molframe", "molgfx")})

data = urllib.request.urlopen("https://files.rcsb.org/download/4HHB.cif").read()
# mmCIF lists only the bonds between residues and ligands; infer the rest so
# ball-and-stick, licorice and lines have sticks to draw.
structure = molframe.read(data, name="4hhb.cif").infer_bonds()
structure, structure.bond_count

## Open the workbench

`Workbench` is a canvas plus a command line. It wraps a `molgfx.Session`, which owns the
names you define (selections, layers) and the undo history, and edits one live scene.

In [ ]:
bench = Workbench(structure)
bench.execute("show cartoon, protein; color chain, @cartoon")
bench

Try typing these in the console under the canvas (press **Enter** to run, **Tab** to complete,
**↑/↓** to recall):

```text
select heme, resname HEM
show spacefill, $heme
select pocket, byres (within 5 of $heme) and protein
show ball_and_stick radius=0.25 as pocket, $pocket
color orange, $pocket
focus $heme
```

Every query is a MolFrame query, compiled by MolFrame from exactly what you wrote.
The same commands can be run from Python:

In [ ]:
result = bench.execute("""
select heme, resname HEM
show spacefill, $heme
select pocket, byres (within 5 of $heme) and protein
show ball_and_stick radius=0.25 as pocket, $pocket
color orange, $pocket
""")
result

## Names follow their definitions

Redefining a selection moves everything that uses it: the `@pocket` layer and the orange
colour rule are retargeted in one small patch; nothing else is rebuilt.

In [ ]:
result = bench.execute("select pocket, byres (within 8 of $heme) and protein")
print(result.messages)
print(result.patch.to_json())

In [ ]:
print(bench.session.explain_layer("pocket"))

## Undo, redo and history

In [ ]:
bench.execute("undo")
print(bench.history)
bench.execute("redo")

## Errors are located and suggest a fix

Nothing changes when any statement of a program fails.

In [ ]:
try:
    bench.execute("show cartoon, protein; hide @pockt")
except molgfx.CommandError as error:
    print(error)
    print(error.errors)

## Typed commands

Applications and agents can build commands without text. They are validated when built.

In [ ]:
from molgfx import Command

bench.session.execute([
    Command.show("licorice", "resname HEM", layer="heme_sticks", bond_radius=0.15),
    Command.color("#ff3366", "@heme_sticks"),
    Command.opacity(0.6, "cartoon"),
])
print(Command.show("surface", "chain A", kind="solvent_accessible", style="mesh"))

## Completion and saved sessions

In [ ]:
print(bench.session.completions("show cartoon, $po"))
saved = bench.session.to_json()
print(saved)

## What you clicked

A click on the canvas picks an atom; the page reports it back to the kernel.

In [ ]:
bench.pick, bench.error